# 1.Package imports section

In [0]:
%pip install openpyxl
dbutils.library.restartPython()

In [0]:
import re
import pandas as pd
import logging

# 2. Dataset configurations and their specific regex patterns

In [0]:

try:
    source_paths = dbutils.fs.ls("/Volumes/cpt_utility_catalog/landing/raw_source_files")
except Exception as e:
    print(f"Error: {e}")

cpt_open_data_portal_datasets = [
    {
        "name": "dam_levels",
        "pattern": r"^dam_levels_.*\.csv$",
        "table": "cpt_utility_catalog.bronze.bronze_dam_levels_raw",
    },
    {
        "name": "substations",
        "pattern": r"^Main_Substations_.*\.csv$",
        "folder":"cpt_open_data_portal",
        "table": "cpt_utility_catalog.bronze.bronze_substations_raw",
    },
    {
        "name": "subcouncils_arrears",
        "pattern": r"^Municipal_Arrears_.*Subcouncils\.csv$",
        "table": "cpt_utility_catalog.bronze.bronze_subcouncils_arrears_raw",
    },
    {
        "name": "suburb_arrears",
        "pattern": r"^Municipal_Arrears_Suburbs.*\.csv$",
        "table": "cpt_utility_catalog.bronze.bronze_suburb_arrears_raw",
    },
    {
        "name": "suburb_electricity_billing",
        "pattern": r"^Suburb_.*Electricity_Billing.*\.csv$",
        "table": "cpt_utility_catalog.bronze.bronze_suburb_electricity_billing_raw",
    },
    {
        "name": "suburb_water_billing",
        "pattern": r"^Suburb_.*Water_Billing.*\.csv$",
        "table": "cpt_utility_catalog.bronze.bronze_suburb_water_billing_raw",
    },
    {
        "name": "service_requests",
        "pattern": r"^Service_Requests_.*\.csv$",
        "table": "cpt_utility_catalog.bronze.bronze_service_requests_raw",
    },
]

stats_sa_datasets = [
    {
        "name": "cpi",
        "pattern": r"^Excel.*CPI.*\.xlsx$",
        "table": "cpt_utility_catalog.bronze.bronze_cpi_raw"
    }
]

# match regex to dataset file
def dataset_matcher(datasets, filename):
    for ds in datasets:
        if re.match(ds["pattern"], filename, re.IGNORECASE):
            matched_ds = ds
            return  matched_ds 

print(f"Loaded {len(cpt_open_data_portal_datasets) + len(stats_sa_datasets)} dataset configurations successfully.")

# 3.Datasets ingestion section

In [0]:
# iterate through sorce folders
for folder in source_paths:
    current_folder_path = f"{folder.path}"
    archive_path = f"{current_folder_path}archive/"

    print(f"checking folder {folder.name}")
    # obtain list of files
    try:
        file_list = dbutils.fs.ls(current_folder_path)
    except Exception as e:
        print(f"Error: {e}")

    print(f"\tFrom the {folder.name} folder, we found the following file(s):")
    # iterate through files
    for file in file_list:
        # remove directories
        if file.isDir():
            continue
        print(f"\t\t- {file.name}")

        # conditional to split files by source folder
        if folder.name == "cpt_open_data_portal/":
            ds_match = dataset_matcher(cpt_open_data_portal_datasets, file.name)

            df_new = (
                spark.read.format("csv")
                .option("header", "true")
                .option("inferSchema", "false")
                .load(file.path)
            )
            print("\t\t\tstatus: dataframe created")

            # write file to bronze layer as delta table
            (df_new.write.format("delta").mode("append").saveAsTable(ds_match["table"]))
            print("\t\t\tstatus: dataframe written as delta table and stored to bronze layer")

            # archive processed files
            (dbutils.fs.mv(file.path, archive_path))
            print(f"\t\t\tstatus: {file.name} archived")

        else:
            ds_match = dataset_matcher(stats_sa_datasets, file.name)

            # # read excel file as pandas dataframe
            # # convert pandas dataframe to spark dataframe
            pd_df = pd.read_excel(file.path[5:], header=0, dtype=str)
            df_new = spark.createDataFrame(pd_df)

            print("\t\t\tstatus: dataframe created")

            # write file to bronze layer as delta table
            (df_new.write.format("delta").mode("append").saveAsTable(ds_match["table"]))
            print("\t\t\tstatus: dataframe written as delta table and stored to bronze layer")

            # archive processed files
            (dbutils.fs.mv(file.path, archive_path))
            print(f"\t\t\tstatus: {file.name} archived")






    